In [ ]:
import subprocess
import time
import statistics
import random

# ============================================================
# CONFIGURATION
# ============================================================

REPETITIONS = 20


# ============================================================
# HELLO WORLD PROGRAMS
# ============================================================

python_hello = '''
print("Hello, World!")
'''

cpp_hello = r'''
#include <iostream>

int main() {
    std::cout << "Hello, World!" << std::endl;
    return 0;
}
'''

with open("hello.py", "w") as f:
    f.write(python_hello)

with open("hello.cpp", "w") as f:
    f.write(cpp_hello)

# Compile C++
subprocess.run(
    ["g++", "-O2", "hello.cpp", "-o", "hello_cpp"],
    check=True
)


# ============================================================
# MERGE SORT - PYTHON
# ============================================================

python_mergesort = '''
import sys

def merge(arr, left, mid, right):
    left_part = arr[left:mid + 1]
    right_part = arr[mid + 1:right + 1]

    i = 0
    j = 0
    k = left

    while i < len(left_part) and j < len(right_part):
        if left_part[i] <= right_part[j]:
            arr[k] = left_part[i]
            i += 1
        else:
            arr[k] = right_part[j]
            j += 1
        k += 1

    while i < len(left_part):
        arr[k] = left_part[i]
        i += 1
        k += 1

    while j < len(right_part):
        arr[k] = right_part[j]
        j += 1
        k += 1


def merge_sort(arr, left, right):
    if left < right:
        mid = (left + right) // 2

        merge_sort(arr, left, mid)
        merge_sort(arr, mid + 1, right)

        merge(arr, left, mid, right)


# Read array from file
with open(sys.argv[1], "r") as f:
    arr = list(map(int, f.read().split()))

merge_sort(arr, 0, len(arr) - 1)
'''

with open("mergesort.py", "w") as f:
    f.write(python_mergesort)


# ============================================================
# MERGE SORT - C++
# ============================================================

cpp_mergesort = r'''
#include <iostream>
#include <fstream>
#include <vector>

using namespace std;

void merge(vector<int>& arr, int left, int mid, int right) {

    vector<int> leftPart(
        arr.begin() + left,
        arr.begin() + mid + 1
    );

    vector<int> rightPart(
        arr.begin() + mid + 1,
        arr.begin() + right + 1
    );

    int i = 0;
    int j = 0;
    int k = left;

    while (i < leftPart.size() && j < rightPart.size()) {

        if (leftPart[i] <= rightPart[j]) {
            arr[k] = leftPart[i];
            i++;
        }
        else {
            arr[k] = rightPart[j];
            j++;
        }

        k++;
    }

    while (i < leftPart.size()) {
        arr[k] = leftPart[i];
        i++;
        k++;
    }

    while (j < rightPart.size()) {
        arr[k] = rightPart[j];
        j++;
        k++;
    }
}


void mergeSort(vector<int>& arr, int left, int right) {

    if (left < right) {

        int mid = left + (right - left) / 2;

        mergeSort(arr, left, mid);
        mergeSort(arr, mid + 1, right);

        merge(arr, left, mid, right);
    }
}


int main(int argc, char* argv[]) {

    ifstream file(argv[1]);

    vector<int> arr;
    int value;

    while (file >> value) {
        arr.push_back(value);
    }

    if (!arr.empty()) {
        mergeSort(arr, 0, arr.size() - 1);
    }

    return 0;
}
'''

with open("mergesort.cpp", "w") as f:
    f.write(cpp_mergesort)

# Compile C++
subprocess.run(
    ["g++", "-O2", "mergesort.cpp", "-o", "mergesort_cpp"],
    check=True
)


# ============================================================
# BENCHMARK FUNCTION
# ============================================================

def benchmark(command, repetitions=20):

    times = []

    # Warm-up
    for _ in range(3):
        subprocess.run(
            command,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

    # Actual measurements
    for _ in range(repetitions):

        start = time.perf_counter()

        subprocess.run(
            command,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        end = time.perf_counter()

        times.append(end - start)

    return times


# ============================================================
# PRINT RESULTS
# ============================================================

def print_results(name1, times1, name2, times2):

    mean1 = statistics.mean(times1)
    mean2 = statistics.mean(times2)

    median1 = statistics.median(times1)
    median2 = statistics.median(times2)

    print("\n" + "=" * 55)
    print("RESULTS")
    print("=" * 55)

    print(f"\n{name1}:")
    print(f"  Mean:   {mean1 * 1000:.4f} ms")
    print(f"  Median: {median1 * 1000:.4f} ms")

    print(f"\n{name2}:")
    print(f"  Mean:   {mean2 * 1000:.4f} ms")
    print(f"  Median: {median2 * 1000:.4f} ms")

    print("\n" + "-" * 55)

    if mean1 > mean2:
        print(f"{name2} is approximately {mean1 / mean2:.2f}x faster")
    else:
        print(f"{name1} is approximately {mean2 / mean1:.2f}x faster")


# ============================================================
# USER MENU
# ============================================================

print("=" * 55)
print("       PYTHON vs C++ PERFORMANCE BENCHMARK")
print("=" * 55)

print("\nChoose the test to execute:")
print()
print("1 - Hello World")
print("2 - Merge Sort")

choice = input("\nYour choice: ")


# ============================================================
# TEST 1 - HELLO WORLD
# ============================================================

if choice == "1":

    print("\nRunning Hello World benchmark...")

    python_times = benchmark(
        ["python3", "hello.py"],
        REPETITIONS
    )

    cpp_times = benchmark(
        ["./hello_cpp"],
        REPETITIONS
    )

    print_results(
        "Python",
        python_times,
        "C++",
        cpp_times
    )


# ============================================================
# TEST 2 - MERGE SORT
# ============================================================

elif choice == "2":

    n = int(input("\nNumber of elements in the array: "))

    if n <= 0:
        print("The number of elements must be greater than 0.")

    else:

        print(f"\nGenerating random array with {n:,} elements...")

        # Same input for Python and C++
        random.seed(42)

        array = [
            random.randint(0, 1_000_000)
            for _ in range(n)
        ]

        # Save array so both programs receive exactly the same data
        with open("array.txt", "w") as f:
            f.write(" ".join(map(str, array)))

        print("Array generated.")
        print(f"Running Merge Sort benchmark ({REPETITIONS} repetitions)...")

        python_times = benchmark(
            ["python3", "mergesort.py", "array.txt"],
            REPETITIONS
        )

        cpp_times = benchmark(
            ["./mergesort_cpp", "array.txt"],
            REPETITIONS
        )

        print_results(
            "Python Merge Sort",
            python_times,
            "C++ Merge Sort",
            cpp_times
        )


# ============================================================
# INVALID CHOICE
# ============================================================

else:
    print("\nInvalid choice. Please select 1 or 2.")

       PYTHON vs C++ PERFORMANCE BENCHMARK

Choose the test to execute:

1 - Hello World
2 - Merge Sort

Your choice: 2

Number of elements in the array: 100000

Generating random array with 100,000 elements...
Array generated.
Running Merge Sort benchmark (20 repetitions)...

RESULTS

Python Merge Sort:
  Mean:   373.6695 ms
  Median: 354.3510 ms

C++ Merge Sort:
  Mean:   18.2603 ms
  Median: 17.9196 ms

-------------------------------------------------------
C++ Merge Sort is approximately 20.46x faster
